In [1]:
suppressPackageStartupMessages(library(tibble))
suppressPackageStartupMessages(library(rWikiPathways))
suppressPackageStartupMessages(library("FactoMineR"))
suppressPackageStartupMessages(library(fgsea))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(stringr))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(GSA))
suppressPackageStartupMessages(library(clustifyr))
suppressPackageStartupMessages(library(fgsea))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(factoextra))
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(grid))
suppressPackageStartupMessages(library(shadowtext))
suppressPackageStartupMessages(library(gridExtra))
suppressPackageStartupMessages(library(lattice))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(GSEABase))
suppressPackageStartupMessages(library(GSVA))
suppressPackageStartupMessages(library(edgeR))
suppressPackageStartupMessages(library(matrixStats))
suppressPackageStartupMessages(library(org.Hs.eg.db))
suppressPackageStartupMessages(library(AnnotationDbi) )
suppressPackageStartupMessages(library(limma))
suppressPackageStartupMessages(library(stringr))
suppressPackageStartupMessages(library(tidyverse))

suppressPackageStartupMessages(library(tictoc))
suppressPackageStartupMessages(library(dplyr))

# Run fGSEA on gene expression data -- gives us what pathways are enriched in disease 

## Endocrine cells -- including HPAP data

In [ ]:
### Run GSEA on each cell type and disease
KEGG_react <- gmtPathways('/reference_files/reactome_kegg.gmt.txt')
cell_type_use = c('Beta','Alpha','Delta')
disease_use <- c('HPAPnPOD.Aab','HPAPnPOD.early','HPAPnPOD.Multiple_Aab','HPAPnPOD.single_Aab','HPAPnPOD.T1D_late','HPAPonly.single_Aab','nPODonly.Multiple_Aab')

for (cell_num in 1:length(cell_type_use)){
    mainDir <- '/downstream_analysis/RNA_fGSEA_fishers'
    subDir <- paste0(cell_type_use[cell_num],'/')
    print(file.path(mainDir, subDir))
    dir.create(file.path(mainDir, subDir))
    
    for (disease_num in 1:length(disease_use)){
        setwd(file.path(mainDir, subDir))

        print(paste0("/downstream_analysis/RNA_deseq/",cell_type_use[cell_num],'.deseq.',disease_use[disease_num],'.tsv'))
        deseq_file <- paste0("/downstream_analysis/RNA_deseq/",cell_type_use[cell_num],'.deseq.',disease_use[disease_num],'.tsv')
        if(!file.exists(deseq_file)) next

        res <- read.table(deseq_file, sep = '\t')
        
        rpl <- fread('/reference_files/rpl_file_gsea.csv', fill=TRUE, header=TRUE)
        rps <- fread('/reference_files/rps_file_gsea.csv', fill=TRUE, header=TRUE)
        mtr <- fread('/reference_files/mts_file_gsea.csv', fill=TRUE, header=TRUE)

        ribo_proteins <- c(rpl$`Approved symbol`, rps$`Approved symbol`, mtr$`Approved symbol`)
        ribo_proteins <- ribo_proteins[which(ribo_proteins != 'Approved symbol')]
        
        res <- res[which(!rownames(res) %in% ribo_proteins),]
        res$rank = (-log10(as.numeric(res$pvalue)))*res$log2FoldChange
        res = data.frame("SYMBOL" = rownames(res),
                         "stat" = res$rank)
        res = res[!grepl(pattern = "NA", x = res$SYMBOL),]
        res = res[!grepl(pattern = "MT-", x = res$SYMBOL),]
        ranks <- deframe(res)
        ranks = sort(ranks, decreasing=TRUE)
        deseq_df <- data.frame(gene = res$SYMBOL, GSEArank = res$stat)
        head(ranks)

        
        KEGG_react_fgseaRes <- fgseaMultilevel(pathways=KEGG_react,
                                    stats=ranks,
                                    eps =0.0,
                                    minSize  = 0, 
                                    maxSize  = 1000)
        message("Number of total enriched terms ", cell_type_use[cell_num], " for ", disease_use[disease_num] ,": ", nrow(KEGG_react_fgseaRes))
        FDR_tresh = 0.10
        KEGG_react_fgseaRes.tresh = KEGG_react_fgseaRes[KEGG_react_fgseaRes$padj < FDR_tresh,]
        message("Number of significant terms: ", cell_type_use[cell_num], " for ", disease_use[disease_num] ,": ",nrow(KEGG_react_fgseaRes.tresh))
        ## Add categories
        res <- KEGG_react_fgseaRes.tresh
        top10_up <- res[which(res$ES > 0),][c(1:10),]
        top10_down <- res[which(res$ES < 0),][c(1:10),]
        top10_up$LOG10P <- -log10(top10_up$pval)
        top10_down$LOG10P <- -log10(top10_down$pval)
        
        KEGG_react_fgseaRes_filt <- unnest(KEGG_react_fgseaRes, leadingEdge)
        KEGG_react_fgseaRes_filt <- as.data.frame(KEGG_react_fgseaRes_filt)
        colnames(KEGG_react_fgseaRes_filt)[8] <- "DESeq_leading_gene"
        
        res_tib <- unnest(res, leadingEdge)
        res_df <- as.data.frame(res_tib)
        colnames(res_df)[8] <- "DESeq_leading_gene"
            
        
        res_filt <- rbind(top10_up,top10_down)
        res_tib_filt <- unnest(res_filt, leadingEdge)
        res_df_filt <- as.data.frame(res_tib_filt)
        colnames(res_df_filt)[8] <- "DESeq_leading_gene"
                
        Out_file1 = paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_res_df.csv')
        write.csv(res_df,Out_file1, quote=FALSE)
        
        Out_file2 = paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_res_df_top10.csv')
        write.csv(res_df_filt,Out_file2, quote=FALSE)
        
        Out_file3 = paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_res_df_allRes_unfilt.csv')
        write.csv(KEGG_react_fgseaRes_filt,Out_file3, quote=FALSE)
    }
}



## All other cell types

In [ ]:
### Run GSEA on each cell type and disease
KEGG_react <- gmtPathways('/reference_files/reactome_kegg.gmt.txt')
cell_type_use = c('Acinar_4','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo', 'Activated_Stellate','Endothelial','Schwann','Ductal','Acinar_3','Acinar_5','Acinar_1_2_6')
disease_use <- c('2.Autoab+','3.T1D_early','4.T1D_late')

for (cell_num in 1:length(cell_type_use)){
    mainDir <- '/downstream_analysis/RNA_fGSEA_fishers'
    subDir <- paste0(cell_type_use[cell_num],'/')
    print(file.path(mainDir, subDir))
    dir.create(file.path(mainDir, subDir))
    
    for (disease_num in 1:length(disease_use)){
        setwd(file.path(mainDir, subDir))

        print(paste0("/downstream_analysis/RNA_deseq/",cell_type_use[cell_num],'.deseq.',disease_use[disease_num],'.tsv'))
        deseq_file <- paste0("/downstream_analysis/RNA_deseq/",cell_type_use[cell_num],'.deseq.',disease_use[disease_num],'.tsv')
        if(!file.exists(deseq_file)) next

        res <- read.table(deseq_file, sep = '\t')
        
        rpl <- fread('/reference_files/rpl_file_gsea.csv', fill=TRUE, header=TRUE)
        rps <- fread('/reference_files/rps_file_gsea.csv', fill=TRUE, header=TRUE)
        mtr <- fread('/reference_files/mts_file_gsea.csv', fill=TRUE, header=TRUE)

        ribo_proteins <- c(rpl$`Approved symbol`, rps$`Approved symbol`, mtr$`Approved symbol`)
        ribo_proteins <- ribo_proteins[which(ribo_proteins != 'Approved symbol')]
        
        res <- res[which(!rownames(res) %in% ribo_proteins),]
        res$rank = (-log10(as.numeric(res$pvalue)))*res$log2FoldChange
        res = data.frame("SYMBOL" = rownames(res),
                         "stat" = res$rank)
        res = res[!grepl(pattern = "NA", x = res$SYMBOL),]
        res = res[!grepl(pattern = "MT-", x = res$SYMBOL),]
        ranks <- deframe(res)
        ranks = sort(ranks, decreasing=TRUE)
        deseq_df <- data.frame(gene = res$SYMBOL, GSEArank = res$stat)
        head(ranks)

        
        KEGG_react_fgseaRes <- fgseaMultilevel(pathways=KEGG_react,
                                    stats=ranks,
                                    eps =0.0,
                                    minSize  = 0, 
                                    maxSize  = 1000)#,nPermSimple = 10000)
        message("Number of total enriched terms ", cell_type_use[cell_num], " for ", disease_use[disease_num] ,": ", nrow(KEGG_react_fgseaRes))
        FDR_tresh = 0.10
        KEGG_react_fgseaRes.tresh = KEGG_react_fgseaRes[KEGG_react_fgseaRes$padj < FDR_tresh,]
        message("Number of significant terms: ", cell_type_use[cell_num], " for ", disease_use[disease_num] ,": ",nrow(KEGG_react_fgseaRes.tresh))
        ## Add categories
        res <- KEGG_react_fgseaRes.tresh
        top10_up <- res[which(res$ES > 0),][c(1:10),]
        top10_down <- res[which(res$ES < 0),][c(1:10),]
        top10_up$LOG10P <- -log10(top10_up$pval)
        top10_down$LOG10P <- -log10(top10_down$pval)
        
        KEGG_react_fgseaRes_filt <- unnest(KEGG_react_fgseaRes, leadingEdge)
        KEGG_react_fgseaRes_filt <- as.data.frame(KEGG_react_fgseaRes_filt)
        colnames(KEGG_react_fgseaRes_filt)[8] <- "DESeq_leading_gene"
        
        res_tib <- unnest(res, leadingEdge)
        res_df <- as.data.frame(res_tib)
        colnames(res_df)[8] <- "DESeq_leading_gene"            
        
        res_filt <- rbind(top10_up,top10_down)
        res_tib_filt <- unnest(res_filt, leadingEdge)
        res_df_filt <- as.data.frame(res_tib_filt)
        colnames(res_df_filt)[8] <- "DESeq_leading_gene"
                
        Out_file1 = paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_res_df.csv')
        write.csv(res_df,Out_file1, quote=FALSE)
        
        Out_file2 = paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_res_df_top10.csv')
        write.csv(res_df_filt,Out_file2, quote=FALSE)
        
        Out_file3 = paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_res_df_allRes_unfilt.csv')
        write.csv(KEGG_react_fgseaRes_filt,Out_file3, quote=FALSE)
    }
}



# Identify what TF-modules are enriched in every pathway using Fisher's test

In [ ]:
### SHIT THAT ONLY NEEDS TO BE READ IN ONCE
gene_units <- gmt_to_list(
  '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/PathwayAnalysis_DESeqResults/reactome_kegg.gmt.txt',
  cutoff = 0,
  sep = "\thttp://www.gsea-msigdb.org/gsea/msigdb/human/geneset/.*?\t"
)

reactome_KEGG_final <- data.frame()

for (i in 1:length(gene_units)){
    reactome <- NULL
    reactome$pathway <- names(gene_units[i])
    for (j in 1:length(gene_units[[i]])){
        reactome$gene <- gene_units[[i]][j]
        reactome_KEGG_final <- rbind(reactome_KEGG_final,reactome)

    }
}
head(reactome_KEGG_final)

## Endocrine cells

In [ ]:
### read in GSEA results, make TF_res, then perform fisher's 
#cell_type_use = c('Acinar_1_2_6','Acinar_3','Acinar_4','Acinar_5','Activated_Stellate','Alpha','Beta','Ductal','Endothelial','Macrophage','Quiescent_Stellate','Tcells')
cell_type_use = c('Alpha','Beta')
#cell_type_use = c('Macrophage','Quiescent_Stellate','Tcells')
#disease_use <- c('HPAPandnPOD.earlyT1D','HPAPnPOD.Aab','HPAPnPOD.T1D_late','HPAPonly.single_Aab','nPODonly.Multiple_Aab','2.Autoab+','3.T1D_early','4.T1D_late')
disease_use <- c('HPAPnPOD.early','HPAPnPOD.Aab','HPAPnPOD.Multiple_Aab','HPAPnPOD.single_Aab','HPAPnPOD.T1D_late','HPAPonly.single_Aab','nPODonly.Multiple_Aab')

for (cell_num in 1:length(cell_type_use)){
    print(paste0("Working on: ",cell_type_use[cell_num]))
    mainDir <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_fishers'
    subDir <- paste0(cell_type_use[cell_num],'/')
    print(file.path(mainDir, subDir))
    dir.create(file.path(mainDir, subDir))

    
 for(disease_num in 1:length(disease_use) ) {
    setwd(file.path(mainDir, subDir))
     
    file_name <- paste0('/nfs/lab/projects/nPOD/downstream_files/GRN/final_results/',cell_type_use[cell_num],'/RNA_fGSEA_noMT_noRibo/',cell_type_use[cell_num],'_',disease_use[disease_num],'_res_df.csv')
    if (file.exists(file_name)) {
        res <- read.table(file_name, sep =',', header= TRUE)
        if (nrow(res) == 0) next 
    
        res$pathway <- gsub('REACTOME_','',res$pathway)
        
        modules <- read.table(paste0('/nfs/lab/projects/nPOD/downstream_files/GRN/',cell_type_use[cell_num],'/',cell_type_use[cell_num],'_GRN_filt_GoodTFs_filt5tpm.txt'), sep = '\t', header=TRUE)
        
        all_gene <- inner_join(reactome_KEGG_final, res, by = "pathway",multiple = "all",relationship = "many-to-many")
        TF_res <- inner_join(modules, all_gene, by = "gene",multiple = "all",relationship = "many-to-many")
        #length(unique(TF_res$pathway))
        
        TFbyGene <- reshape2::dcast(TF_res, TF ~ gene, value.var = "gene", fun.aggregate = length,fill = 0)
        rownames(TFbyGene) <- TFbyGene$TF
        TFbyGene[1] <- NULL 
        TFbyGene[TFbyGene > 0] <- 1
        options(repr.plot.width=10, repr.plot.height=10)
    
        hist(colSums(TFbyGene), main = paste0("TF per gene for: ",cell_type_use[cell_num]), xlab ='Num of TFs')
        print(paste0("Mean TF per gene for ",cell_type_use[cell_num], " : ", mean(colSums(TFbyGene))))
        if (length(unique(TF_res$pathway)) ==1) next 
        
        GenebyPathway <- reshape2::dcast(TF_res, gene ~ pathway, value.var = "gene", fun.aggregate = length, fill = 0)
        rownames(GenebyPathway) <- GenebyPathway$gene
        GenebyPathway <- GenebyPathway[ , !names(GenebyPathway) %in%  c("gene")]
        GenebyPathway[GenebyPathway > 0] <- 1
        
        
        Pathway <- GenebyPathway
        TF <- TFbyGene
        hist(colSums(GenebyPathway), main = paste0("Genes per Pathway for: ",cell_type_use[cell_num]), xlab ='Num of Genes')
        print(paste0("Mean Genes per Pathway for ",cell_type_use[cell_num], " : ", mean(colSums(GenebyPathway))))
    
        TF <- as.data.frame(t(TF))
        
        #Check for mistmatch
        sum(rownames(Pathway)!=rownames(TF)) 
        rownames(Pathway)[(rownames(Pathway)!=rownames(TF))]
        rownames(TF)[(rownames(Pathway)!=rownames(TF))] 
        
        #Creat matrix to store output
        p_values <- matrix(data=0, nrow=length(colnames(Pathway)),ncol=length(colnames(TF))) 
        rownames(p_values) <- colnames(Pathway)
        colnames(p_values) <- colnames(TF)
        p_values <- as.data.frame(p_values)
        
        
        or_values <- matrix(data=0, nrow=length(colnames(Pathway)),ncol=length(colnames(TF))) 
        rownames(or_values) <- colnames(Pathway)
        colnames(or_values) <- colnames(TF)
        or_values <- as.data.frame(or_values)
        
      ### Run Fishers
      tic()
      #Run tests with nested for loops
      for (i in 1:length(colnames(Pathway))) {
          path <- colnames(Pathway)[i]
          
          for (j in 1:length(colnames(TF))) {
              tf <- colnames(TF)[j]
              test_table <- table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))
              if (dim(test_table)[1] != 2 | dim(test_table)[2] != 2){
                  p_values[path,tf] <- NA
                  or_values[path,tf] <- NA
              } else if(test_table[2,2] == 0 |test_table[2,2] == 1 ){
                  p_values[path,tf] <- NA
                  or_values[path,tf] <- NA
              } else{
                 
              p_values[path,tf] <-  broom::tidy(fisher.test(table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))))$p.value
              or_values[path,tf] <- broom::tidy(fisher.test(table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))))$estimate
                  }
          }
      }
      toc()
      pval_outFile <- paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_pValues_unfilt.txt')
      write.table(p_values, file = pval_outFile, sep ='\t', col.names = TRUE, quote=FALSE)
      
      OR_outFile <- paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_orValues_unfilt.txt')
      write.table(or_values, file = OR_outFile, sep ='\t', col.names = TRUE, quote=FALSE)
      
      hist(unlist(p_values), main = paste0("All P-values for: ",cell_type_use[cell_num]), xlab = 'P values')
      
      fdr_df <- matrix(nrow = nrow(p_values), ncol = ncol(p_values))
      rownames(fdr_df) <- rownames(p_values)
      colnames(fdr_df) <- colnames(p_values)
  
      for (i in 1:ncol(p_values)){
          fdr <- p.adjust(p_values[,i], method = 'fdr')
          fdr_df[,i] <- fdr
      }
      fdr_df <- as.data.frame(fdr_df)
      
      hist(unlist(fdr_df), main = paste0("All FDR for: ",cell_type_use[cell_num]), xlab = 'FDR values')
  
      #### organize results
      res_mod <- data.frame(pathway = TF_res$pathway,
                        TF = TF_res$TF,
                      genes = TF_res$gene, pval = TF_res$pval, padj = TF_res$padj, ES = TF_res$ES)
      aggregated <- aggregate(cbind(genes) ~ pathway + TF + pval +padj + ES , data = res_mod,
                          FUN = function(x) list(unique(x)))
      aggregated$fisher_pval <- NULL
      aggregated$fisher_FDR <- NULL
  
  
      for(i in 1:nrow(aggregated)){
          pathway <- aggregated$pathway[i]
          TF <- aggregated$TF[i]
          fdr <- fdr_df[pathway,TF]
          pval <- p_values[pathway,TF]
          aggregated$fisher_FDR[i] <- fdr
          aggregated$fisher_pval[i] <- pval
  
      }                      
      aggregated <- na.omit(aggregated)
      aggregated$genes <- sapply(aggregated$genes, function(x) paste(x, collapse = ", "))
  
      final_outFile <- paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_TFModules_DrivingPathways_FinalResults_omitNA_noFDRfilt.txt')
      write.table(aggregated, file = final_outFile, sep ='\t', col.names = TRUE, quote=FALSE)
  } else {
  # File does not exist, skip to the next iteration
      print(paste("File does not exist:", file_name))
      next
        }
   }
}
    

## all other cells

In [ ]:
### read in GSEA results, make TF_res, then perform fisher's 
cell_type_use = c('Acinar_4','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo',
           'Activated_Stellate','Endothelial','Schwann','Ductal','Acinar_3','Acinar_5','Acinar_1_2_6')
disease_use <- c('2.Autoab+','3.T1D_early','4.T1D_late')

for (cell_num in 1:length(cell_type_use)){
    print(paste0("Working on: ",cell_type_use[cell_num]))
    mainDir <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_fishers'
    subDir <- paste0(cell_type_use[cell_num],'/')
    print(file.path(mainDir, subDir))
    dir.create(file.path(mainDir, subDir))
    modules <- paste0('/nfs/lab/projects/nPOD/downstream_files/GRN/',cell_type_use[cell_num],'/',cell_type_use[cell_num],'_GRN_filt_GoodTFs_filt5tpm.txt')
    if (file.exists(modules)) {

        modules <- read.table(paste0('/nfs/lab/projects/nPOD/downstream_files/GRN/',cell_type_use[cell_num],'/',cell_type_use[cell_num],'_GRN_filt_GoodTFs_filt5tpm.txt'), sep = '\t', header=TRUE)

        for(disease_num in 1:length(disease_use) ) {
        setwd(file.path(mainDir, subDir))
        file_name <- paste0('/nfs/lab/projects/nPOD/downstream_files/GRN/final_results/',cell_type_use[cell_num],'/RNA_fGSEA_noMT_noRibo/',cell_type_use[cell_num],'_',disease_use[disease_num],'_res_df.csv')
        if (file.exists(file_name)) {
         res <- read.table(file_name, sep =',', header= TRUE)
         if (nrow(res) == 0) next 
     
            res$pathway <- gsub('REACTOME_','',res$pathway)
            
            
            all_gene <- inner_join(reactome_KEGG_final, res, by = "pathway",multiple = "all",relationship = "many-to-many")
            TF_res <- inner_join(modules, all_gene, by = "gene",multiple = "all",relationship = "many-to-many")
            #length(unique(TF_res$pathway))
            
            TFbyGene <- reshape2::dcast(TF_res, TF ~ gene, value.var = "gene", fun.aggregate = length,fill = 0)
            rownames(TFbyGene) <- TFbyGene$TF
            TFbyGene[1] <- NULL 
            TFbyGene[TFbyGene > 0] <- 1
            options(repr.plot.width=10, repr.plot.height=10)
        
            hist(colSums(TFbyGene), main = paste0("TF per gene for: ",cell_type_use[cell_num]), xlab ='Num of TFs')
            print(paste0("Mean TF per gene for ",cell_type_use[cell_num], " : ", mean(colSums(TFbyGene))))
            if (length(unique(TF_res$pathway)) ==1) next 
            
            GenebyPathway <- reshape2::dcast(TF_res, gene ~ pathway, value.var = "gene", fun.aggregate = length, fill = 0)
            rownames(GenebyPathway) <- GenebyPathway$gene
            GenebyPathway <- GenebyPathway[ , !names(GenebyPathway) %in%  c("gene")]
            GenebyPathway[GenebyPathway > 0] <- 1
            
            
            Pathway <- GenebyPathway
            TF <- TFbyGene
            hist(colSums(GenebyPathway), main = paste0("Genes per Pathway for: ",cell_type_use[cell_num]), xlab ='Num of Genes')
            print(paste0("Mean Genes per Pathway for ",cell_type_use[cell_num], " : ", mean(colSums(GenebyPathway))))
        
            TF <- as.data.frame(t(TF))
            
            #Check for mistmatch
            sum(rownames(Pathway)!=rownames(TF)) 
            rownames(Pathway)[(rownames(Pathway)!=rownames(TF))]
            rownames(TF)[(rownames(Pathway)!=rownames(TF))] 
            
            #Creat matrix to store output
            p_values <- matrix(data=0, nrow=length(colnames(Pathway)),ncol=length(colnames(TF))) 
            rownames(p_values) <- colnames(Pathway)
            colnames(p_values) <- colnames(TF)
            p_values <- as.data.frame(p_values)
            
            
            or_values <- matrix(data=0, nrow=length(colnames(Pathway)),ncol=length(colnames(TF))) 
            rownames(or_values) <- colnames(Pathway)
            colnames(or_values) <- colnames(TF)
            or_values <- as.data.frame(or_values)
            
          ### Run Fishers
          tic()
          #Run tests with nested for loops
          for (i in 1:length(colnames(Pathway))) {
              path <- colnames(Pathway)[i]
              
              for (j in 1:length(colnames(TF))) {
                  tf <- colnames(TF)[j]
                  test_table <- table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))
                  if (dim(test_table)[1] != 2 | dim(test_table)[2] != 2){
                      p_values[path,tf] <- NA
                      or_values[path,tf] <- NA
                  } else if(test_table[2,2] == 0 |test_table[2,2] == 1 ){
                      p_values[path,tf] <- NA
                      or_values[path,tf] <- NA
                  } else{
                     
                  p_values[path,tf] <-  broom::tidy(fisher.test(table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))))$p.value
                  or_values[path,tf] <- broom::tidy(fisher.test(table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))))$estimate
                      }
              }
          }
          toc()
          pval_outFile <- paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_pValues_unfilt.txt')
          write.table(p_values, file = pval_outFile, sep ='\t', col.names = TRUE, quote=FALSE)
          
          OR_outFile <- paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_orValues_unfilt.txt')
          write.table(or_values, file = OR_outFile, sep ='\t', col.names = TRUE, quote=FALSE)
          
          hist(unlist(p_values), main = paste0("All P-values for: ",cell_type_use[cell_num]), xlab = 'P values')
          
          fdr_df <- matrix(nrow = nrow(p_values), ncol = ncol(p_values))
          rownames(fdr_df) <- rownames(p_values)
          colnames(fdr_df) <- colnames(p_values)
      
          for (i in 1:ncol(p_values)){
              fdr <- p.adjust(p_values[,i], method = 'fdr')
              fdr_df[,i] <- fdr
          }
          fdr_df <- as.data.frame(fdr_df)
          
          hist(unlist(fdr_df), main = paste0("All FDR for: ",cell_type_use[cell_num]), xlab = 'FDR values')
      
          #### organize results
          res_mod <- data.frame(pathway = TF_res$pathway,
                            TF = TF_res$TF,
                          genes = TF_res$gene, pval = TF_res$pval, padj = TF_res$padj, ES = TF_res$ES)
          aggregated <- aggregate(cbind(genes) ~ pathway + TF + pval +padj + ES , data = res_mod,
                              FUN = function(x) list(unique(x)))
          aggregated$fisher_pval <- NULL
          aggregated$fisher_FDR <- NULL
      
      
          for(i in 1:nrow(aggregated)){
              pathway <- aggregated$pathway[i]
              TF <- aggregated$TF[i]
              fdr <- fdr_df[pathway,TF]
              pval <- p_values[pathway,TF]
              aggregated$fisher_FDR[i] <- fdr
              aggregated$fisher_pval[i] <- pval
      
          }                      
          aggregated <- na.omit(aggregated)
          aggregated$genes <- sapply(aggregated$genes, function(x) paste(x, collapse = ", "))
      
          final_outFile <- paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_TFModules_DrivingPathways_FinalResults_omitNA_noFDRfilt.txt')
          write.table(aggregated, file = final_outFile, sep ='\t', col.names = TRUE, quote=FALSE)
  } else {
  # File does not exist, skip to the next iteration
      print(paste("File does not exist:", file_name))
      next
        }
   }
}
                                    }
    

In [ ]:
###. make excel spreadsheet of fisher results
library(readr)
library(openxlsx)
library(dplyr)
# Directory containing the TSV files
directory <- "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_fishers"

# List all TSV files in the directory
files <- list.files(directory, pattern = "\\TFModules_DrivingPathways_FinalResults_omitNA_noFDRfilt.txt", full.names = TRUE, recursive = TRUE)
#files
# Create an Excel workbook
workbook <- createWorkbook()

# Iterate through each TSV file
for (file in files) {
  # Extract the file name without extension
  sheet_name <- tools::file_path_sans_ext(basename(file))
  split_string <- strsplit(sheet_name, "\\_TFModules_DrivingPathways_FinalResults_omitNA_noFDRfilt")

  # Keep everything before the period
  result1 <- split_string[[1]][1]
      result2 <- split_string[[1]][2]

  result <- split_string#paste0(result1,"_",result2)
  # Read the TSV file
  data <- read.table(file, sep = '\t', header = TRUE)
    #test2 <- subset(data, select = -c(pval,padj,ES))
  sig <- data[data$padj <0.1,]
      sig2 <- sig[sig$fisher_FDR <0.1,]

            sig_path <- length(unique(sig2$pathway))
                sig_TF <- length(unique(sig2$TF))

            print(paste0("Sig results for ",split_string, " is ", sig_path, " pathways and ",sig_TF," TFs"))

  # Add the data as a new sheet in the workbook
  addWorksheet(workbook, result)
  writeData(workbook, result, sig2)
}

# Save the workbook as an Excel file
#saveWorkbook(workbook, "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/tables/Supplementary Table 11_12_remake_sigFilt.xlsx", overwrite = TRUE)


In [ ]:
# Save the workbook as an Excel file
saveWorkbook(workbook, "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/tables/may27/NEW_fGSEA_fishers_filteredTPMGRNS.xlsx", overwrite = TRUE)


# Run all TF GRNs vs all Pathways 

In [ ]:
setwd('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/')


In [ ]:
#'Alpha','Beta','Acinar_4','Macrophage','Tcells',
cell_type_use = c('Quiescent_Stellate','LymphEndo',
           'Activated_Stellate','Endothelial','Schwann','Ductal','Acinar_3','Acinar_5','Acinar_1_2_6')
dir.create('Fishers_allPaths_allTFs')
for (cell_num in 1:length(cell_type_use)){
    modules_file <- paste0('/nfs/lab/projects/nPOD/downstream_files/GRN/',cell_type_use[cell_num],'/',cell_type_use[cell_num],'_GRN_filt_GoodTFs_filt5tpm.txt')
   if(file.exists(modules_file)){
       modules <- read.table(modules_file, sep = '\t', header=TRUE)
        
    #all_gene <- inner_join(reactome_KEGG_final, res, by = "pathway",multiple = "all")
    TF_res <- inner_join(modules, reactome_KEGG_final, by = "gene",multiple = "all",relationship = "many-to-many") 
    TFbyGene <- reshape2::dcast(TF_res, TF ~ gene, value.var = "gene", fun.aggregate = length,fill = 0)
    
    rownames(TFbyGene) <- TFbyGene$TF
    TFbyGene[1] <- NULL 
    TFbyGene[TFbyGene > 0] <- 1
    
    GenebyPathway <- reshape2::dcast(TF_res, gene ~ pathway, value.var = "gene", fun.aggregate = length, fill = 0)
    rownames(GenebyPathway) <- GenebyPathway$gene
    GenebyPathway <- GenebyPathway[ , !names(GenebyPathway) %in%  c("gene")]
    GenebyPathway[GenebyPathway > 0] <- 1
    
    Pathway <- GenebyPathway
    TF <- TFbyGene
    
    TF <- as.data.frame(t(TF))
    
    #Check for mistmatch
    sum(rownames(Pathway)!=rownames(TF)) 
    rownames(Pathway)[(rownames(Pathway)!=rownames(TF))]
    rownames(TF)[(rownames(Pathway)!=rownames(TF))] 
    
    #Creat matrix to store output
    p_values <- matrix(data=0, nrow=length(colnames(Pathway)),ncol=length(colnames(TF))) 
    rownames(p_values) <- colnames(Pathway)
    colnames(p_values) <- colnames(TF)
    p_values <- as.data.frame(p_values)
    
    
    or_values <- matrix(data=0, nrow=length(colnames(Pathway)),ncol=length(colnames(TF))) 
    rownames(or_values) <- colnames(Pathway)
    colnames(or_values) <- colnames(TF)
    or_values <- as.data.frame(or_values)
    
    ### Run Fishers
    tic()
    #Run tests with nested for loops
    for (i in 1:length(colnames(Pathway))) {
        path <- colnames(Pathway)[i]
        
        for (j in 1:length(colnames(TF))) {
            tf <- colnames(TF)[j]
            test_table <- table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))
            if (dim(test_table)[1] != 2 | dim(test_table)[2] != 2){
                p_values[path,tf] <- NA
                or_values[path,tf] <- NA
            } else if(test_table[2,2] == 0 |test_table[2,2] == 1 ){
                p_values[path,tf] <- NA
                or_values[path,tf] <- NA
            } else{
               
            p_values[path,tf] <-  broom::tidy(fisher.test(table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))))$p.value
            or_values[path,tf] <- broom::tidy(fisher.test(table(data.frame(pathway = Pathway[[path]], TF = TF[[tf]]))))$estimate
                }
        }
    }
    toc()
    pval_outFile <- paste0("Fishers_allPaths_allTFs/",cell_type_use[cell_num],"_pValues_unfilt.txt")
    write.table(p_values, file = pval_outFile, sep ='\t', col.names = TRUE, quote=FALSE)
    
    OR_outFile <- paste0("Fishers_allPaths_allTFs/",cell_type_use[cell_num],'_orValues_unfilt.txt')
    write.table(or_values, file = OR_outFile, sep ='\t', col.names = TRUE, quote=FALSE)
    
    #print(hist(unlist(p_values), main = paste0("All P-values for: ",cell_type_use[cell_num]), xlab = 'P values'))
    
    fdr_df <- matrix(nrow = nrow(p_values), ncol = ncol(p_values))
    rownames(fdr_df) <- rownames(p_values)
    colnames(fdr_df) <- colnames(p_values)
    
    for (i in 1:ncol(p_values)){
        fdr <- p.adjust(p_values[,i], method = 'fdr')
        fdr_df[,i] <- fdr
    }
    fdr_df <- as.data.frame(fdr_df)
    
    #hist(unlist(fdr_df), main = paste0("All FDR for: ",cell_type_use[cell_num]), xlab = 'FDR values')
    
    #### organize results
    res_mod <- data.frame(pathway = TF_res$pathway,
                      TF = TF_res$TF,
                    genes = TF_res$gene)
    aggregated <- aggregate(cbind(genes) ~ pathway + TF , data = res_mod,
                        FUN = function(x) list(unique(x)))
    aggregated$fisher_pval <- NULL
    aggregated$fisher_FDR <- NULL
    
    
    for(i in 1:nrow(aggregated)){
        pathway <- aggregated$pathway[i]
        TF <- aggregated$TF[i]
        fdr <- fdr_df[pathway,TF]
        pval <- p_values[pathway,TF]
        aggregated$fisher_FDR[i] <- fdr
        aggregated$fisher_pval[i] <- pval
    
    }                      
    aggregated <- na.omit(aggregated)
    aggregated$genes <- sapply(aggregated$genes, function(x) paste(x, collapse = ", "))
    
    final_outFile <- paste0("Fishers_allPaths_allTFs/",cell_type_use[cell_num],'_TFModules_FinalResults_omitNA_noFDRfilt.txt')
    write.table(aggregated, file = final_outFile, sep ='\t', col.names = TRUE, quote=FALSE)

   }
    

    }

# Moving on to chromatin GRN stuff

In [ ]:
setwd('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/')


## run fGSEA TF as pathways, peaks as input -- using logistic regression results 

In [ ]:
### first make GMT file using final GRN TF-CREs units
cell_type_use = c('Acinar_1_2_6','Acinar_3','Acinar_4','Acinar_5','Activated_Stellate','Alpha','Beta','Ductal','Endothelial','Macrophage','Quiescent_Stellate','Tcells')
#cell_type_use <- 'Beta'
for (cell_num in 1:length(cell_type_use)){
    print(paste0("Working on: ",cell_type_use[cell_num]))
    mainDir <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks'
    subDir <- paste0(cell_type_use[cell_num],'/')
    print(file.path(mainDir, subDir))
    dir.create(file.path(mainDir, subDir))

    setwd(file.path(mainDir, subDir))
    ### create GMT
    modules <- read.table(paste0('/nfs/lab/projects/nPOD/downstream_files/GRN/',cell_type_use[cell_num],'/',cell_type_use[cell_num],'_GRN_filt_GoodTFs_filt5tpm.txt'), sep = '\t', header=TRUE)
    gmt_tfModules<- data.frame(id=modules$TF, name = cell_type_use[cell_num], gene = modules$peak)                         
    gmt_tfModules <- gmt_tfModules[!duplicated(gmt_tfModules),]
    thedate <- strftime(Sys.Date(),"%y%m%d")
    CELL <- cell_type_use[cell_num]
    GMT_pathName <- paste0(CELL,"_GSEA_GMT_TF_modules_cCREs_noDup.gmt",sep='')
    writeGMT(gmt_tfModules, GMT_pathName)
    
}

In [ ]:
#### then we need to intersect the cell type specific peaks used in the logistic regression with the fixed 
#### peaks used for the GRN construction, this will allow use to test for TF module enrichment in disease
#### based off of changes in chromatin accessibilty

## first read in all TF - CRE links
TF_peak_gene <- read.table('/nfs/lab/rlmelton/npod/GRN/T1D_GRN/TF2peak2gene/TF_peak_gene.csv', sep=',', header =  TRUE)
TF_peak_gene$peak <- gsub(':','_',TF_peak_gene$peak)
TF_peak_gene$peak <- gsub('-','_',TF_peak_gene$peak)
TF_peak_gene[c('chr', 'start','end')] <- str_split_fixed(TF_peak_gene$peak, '_', 3)
my_data2 <- TF_peak_gene[, c('chr', 'start','end','TF','peak',
                             'gene','cell_type','score_distance','type')]
head(my_data2)


In [ ]:
celltype = c('Acinar_1_2_6','Acinar_3','Acinar_4','Acinar_5','Activated_Stellate','Alpha','Beta','Ductal','Endothelial','Macrophage','Quiescent_Stellate','Tcells')
for (CELL in celltype){
    mainDir <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks'
    subDir <- paste0(CELL,'/')
    setwd(file.path(mainDir, subDir))
    
    TF_peak_gene_ct <-  my_data2[my_data2$cell_type == c(CELL,'all'),]
    grn_file <- sprintf('/nfs/lab/projects/nPOD/downstream_files/GRN/%s/%s_GRN_filt_GoodTFs_filt5tpm.txt',CELL,CELL)
    ct_GRN <- read.table(grn_file, sep = '\t', header = TRUE, check.names = FALSE)
    
    TF_peak_gene_ct_tfFilt <-  TF_peak_gene_ct[TF_peak_gene_ct$TF %in% ct_GRN$TF,]

    write.table(TF_peak_gene_ct_tfFilt,paste0(CELL,'_TF_peak_gene.bed'), sep='\t', quote=FALSE, col.names = FALSE, row.names = FALSE)
   # print(sprintf("bedtools intersect -a /nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/%s_peaks.bed -b /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/%s_TF_peak_gene.bed -wa -wb  > /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/%s_TF_peak_gene_bedtools.bed", CELL, CELL,CELL,CELL,CELL))
}

## Remake gmt files

In [ ]:
gene_units <- gmt_to_list(
  '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/PathwayAnalysis_DESeqResults/reactome_kegg.gmt.txt',
  cutoff = 0,
  sep = "\thttp://www.gsea-msigdb.org/gsea/msigdb/human/geneset/.*?\t"
)

In [ ]:
reactome_KEGG_final <- data.frame()

for (i in 1:length(gene_units)){
    reactome <- NULL
    reactome$pathway <- names(gene_units[i])
    for (j in 1:length(gene_units[[i]])){
        reactome$gene <- gene_units[[i]][j]
        reactome_KEGG_final <- rbind(reactome_KEGG_final,reactome)

    }
}
head(reactome_KEGG_final)

In [ ]:
celltype = c('Acinar_1_2_6','Acinar_3','Acinar_4','Acinar_5','Activated_Stellate','Alpha','Beta','Ductal','Endothelial','Macrophage','Quiescent_Stellate','Tcells')
for (CELL in celltype){
    TF_peak_gene_bedtools <- read.table(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/%s_TF_peak_gene_bedtools.bed',CELL,CELL),sep='\t')
    TF_peak_gene_bedtools_mod <- TF_peak_gene_bedtools[,colnames(TF_peak_gene_bedtools) %in% c('V1','V2','V3','V7','V8','V9','V10','V11','V12')]
    TF_peak_gene_bedtools_mod$peak <- paste0(TF_peak_gene_bedtools_mod$V1, ':',TF_peak_gene_bedtools_mod$V2,'-',TF_peak_gene_bedtools_mod$V3)
    colnames(TF_peak_gene_bedtools_mod) <- c('chrom','start','end','TF','peak_og','gene','cell_type',
                                        'score','source','peak')
    TF_peak_gene <- TF_peak_gene_bedtools_mod[,colnames(TF_peak_gene_bedtools_mod) %in% c('TF','gene','cell_type',
                                            'score','source','peak')]
    
    TF_peak_gene_activeCelltype <- TF_peak_gene[TF_peak_gene$cell_type == c(CELL,'all'),]
    path2genes2peaks<- inner_join(reactome_KEGG_final, TF_peak_gene_activeCelltype, by = "gene",multiple = "all",relationship = "many-to-many")
    df_final <- na.omit(path2genes2peaks)
    beta_GSEA_grn_sub <- data.frame(id=df_final$pathway, peak = df_final$peak) 
    beta_GSEA_grn_sub <- beta_GSEA_grn_sub[!duplicated(beta_GSEA_grn_sub),]
    beta_GSEA_grn_sub$peak <- gsub(':','_',beta_GSEA_grn_sub$peak)
    beta_GSEA_grn_sub$peak <- gsub('-','_',beta_GSEA_grn_sub$peak)
    head(beta_GSEA_grn_sub)  
    dim(beta_GSEA_grn_sub)
    thedate <- strftime(Sys.Date(),"%y%m%d")
    CELL #<- cell_type_use[cell_num]
    gmtPath <- sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/',CELL)
    GMT_pathName <- paste0(gmtPath,CELL,"_GSEA_GMT_Path2celltypeSpecificPeaks_cCREs_noDup_",thedate,'.gmt',sep='')
    writeGMT(beta_GSEA_grn_sub, GMT_pathName)
    }

# fGSEA path2peaks

In [ ]:
setwd('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_path2peaks')

In [ ]:
cell_type_use = c('Acinar_1_2_6','Acinar_3','Acinar_4','Acinar_5','Activated_Stellate','Alpha','Beta','Ductal','Endothelial','Macrophage','Quiescent_Stellate','Tcells')
disease_use <- c('Aab','earlyT1D','lateT1D')


for (cell_num in 1:length(cell_type_use)){
    print(paste0("Working on: ",cell_type_use[cell_num]))
    mainDir <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_path2peaks'
    subDir <- paste0(cell_type_use[cell_num],'_GSEA_path2peak/')
    print(file.path(mainDir, subDir))
    dir.create(file.path(mainDir, subDir))

    setwd(file.path(mainDir, subDir))
    ### read in specific GMT
    gmtPath <- sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/',CELL)
    KEGG_react <- gmtPathways(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/%s_GSEA_GMT_Path2celltypeSpecificPeaks_cCREs_noDup_240731.gmt',cell_type_use[cell_num],cell_type_use[cell_num]))

    for (disease_num in 1:length(disease_use)){
        file_name <- paste0(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724/071824_recalculateLogFC_DiseaseDiff/%s_%s.LogRegOut_recalcFC.txt', cell_type_use[cell_num],disease_use[disease_num]))
        if (file.exists(file_name)) {
            print(file_name)
            res <- read.table(file_name, sep = '\t', header = TRUE)
            pval_col <- paste0('pvalue_',disease_use[disease_num])
            if(pval_col %in% colnames(res)){
                logfc_col <- 'L2FC'
                res$rank = (-log10(as.numeric(res[,pval_col])))*res[,logfc_col]
                res <- res %>% 
                          filter_all(all_vars(!is.infinite(.)))
                res = data.frame("SYMBOL" = res$peak,
                             "stat" = res$rank)
                #res$SYMBOL <- paste("chr", res$SYMBOL, sep = "")
    
                res = res[!grepl(pattern = "NA", x = res$SYMBOL),]
                ranks <- deframe(res)
                ranks = sort(ranks, decreasing=TRUE)
                deseq_df <- data.frame(gene = res$SYMBOL, GSEArank = res$stat)
                KEGG_react_fgseaRes <- fgseaMultilevel(pathways=KEGG_react,
                                       stats=ranks,
                                       eps =0.0,
                                       minSize  = 0)#,nPermSimple = 10000)
               message("Number of total enriched terms ", cell_type_use[cell_num], " for ", disease_use[disease_num] ,": ", nrow(KEGG_react_fgseaRes))
               FDR_tresh = 0.10
               KEGG_react_fgseaRes.tresh = KEGG_react_fgseaRes[KEGG_react_fgseaRes$padj < FDR_tresh,]
               message("Number of significant terms: ", cell_type_use[cell_num], ": ",nrow(KEGG_react_fgseaRes.tresh))
               ## Add categories
               res <- KEGG_react_fgseaRes.tresh
               top10_up <- res[which(res$ES > 0),][c(1:10),]
               top10_down <- res[which(res$ES < 0),][c(1:10),]
               top10_up$LOG10P <- -log10(top10_up$pval)
               top10_down$LOG10P <- -log10(top10_down$pval)
               outfile_all <-  paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_fgsea_allResults.tsv')
               KEGG_react_fgseaRes %>% 
                   rowwise() %>% 
                   mutate_if(is.list, ~paste(unlist(.), collapse = ',')) %>% 
                   write.table(outfile_all,sep='\t',quote = FALSE, row.names = FALSE)
               if (nrow(res) > 0){
                   outfile_sig <-  paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_fgsea_sigResults.tsv')
                   res %>% 
                       rowwise() %>% 
                       mutate_if(is.list, ~paste(unlist(.), collapse = ',')) %>% 
                       write.table(outfile_sig,sep='\t',quote = FALSE, row.names = FALSE)
                    }
            } else next
        
                
    }
         else {
    # File does not exist, skip to the next iteration
        print(paste("File does not exist:", file_name))
        next
          }
    }
    setwd('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_path2peaks')
}

# fGSEA -- TF2peaks

In [ ]:
for (CELL in celltype){
    TF_peak_gene_bedtools <- read.table(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/%s_TF_peak_gene_bedtools.bed',CELL,CELL),sep='\t')
    TF_peak_gene_bedtools_mod <- TF_peak_gene_bedtools[,colnames(TF_peak_gene_bedtools) %in% c('V1','V2','V3','V7','V8','V9','V10','V11','V12')]
    TF_peak_gene_bedtools_mod$peak <- paste0(TF_peak_gene_bedtools_mod$V1, ':',TF_peak_gene_bedtools_mod$V2,'-',TF_peak_gene_bedtools_mod$V3)
    colnames(TF_peak_gene_bedtools_mod) <- c('chrom','start','end','TF','peak_og','gene','cell_type',
                                        'score','source','peak')
    TF_peak_gene <- TF_peak_gene_bedtools_mod[,colnames(TF_peak_gene_bedtools_mod) %in% c('TF','gene','cell_type',
                                            'score','source','peak')]
   TF_peak_gene_activeCelltype <- TF_peak_gene[TF_peak_gene$cell_type == c(CELL,'all'),]

   beta_GSEA_grn_sub <- data.frame(id=TF_peak_gene_activeCelltype$T, peak = TF_peak_gene_activeCelltype$peak) 
   beta_GSEA_grn_sub <- beta_GSEA_grn_sub[!duplicated(beta_GSEA_grn_sub),]
   beta_GSEA_grn_sub$peak <- gsub(':','_',beta_GSEA_grn_sub$peak)
   beta_GSEA_grn_sub$peak <- gsub('-','_',beta_GSEA_grn_sub$peak)

   thedate <- strftime(Sys.Date(),"%y%m%d")
   CELL #<- cell_type_use[cell_num]
   gmtPath <- sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/',CELL)
   GMT_pathName <- paste0(gmtPath,CELL,"_GSEA_GMT_TF2celltypeSpecificPeaks_cCREs_noDup_",thedate,'.gmt',sep='')
   writeGMT(beta_GSEA_grn_sub, GMT_pathName)
    }

In [ ]:
setwd('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TF2peaks/')

In [ ]:
cell_type_use = c('Acinar_1_2_6','Acinar_3','Acinar_4','Acinar_5','Activated_Stellate','Alpha','Beta','Ductal','Endothelial','Macrophage','Quiescent_Stellate','Tcells')
disease_use <- c('Aab','earlyT1D','lateT1D')
#cell_type_use = c('Acinar_1_2_6','Acinar_3','Acinar_4','Acinar_5','Activated_Stellate','Alpha','Beta','Ductal','Endothelial','Macrophage','Quiescent_Stellate','Tcells')
#disease_use <- c('02_Aab','03_earlyT1D','04_lateT1D')

for (cell_num in 1:length(cell_type_use)){
    print(paste0("Working on: ",cell_type_use[cell_num]))
    mainDir <- '/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TF2peaks'
    subDir <- paste0(cell_type_use[cell_num],'_GSEA_TF2peak/')
    print(file.path(mainDir, subDir))
    dir.create(file.path(mainDir, subDir))

    setwd(file.path(mainDir, subDir))
    ### read in specific GMT
    gmtPath <- sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/',CELL)
    KEGG_react <- gmtPathways(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TFmodules_InputPeaks/%s/%s_GSEA_GMT_TF2celltypeSpecificPeaks_cCREs_noDup_240731.gmt',cell_type_use[cell_num],cell_type_use[cell_num]))

    for (disease_num in 1:length(disease_use)){
        file_name <- paste0(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724/071824_recalculateLogFC_DiseaseDiff/%s_%s.LogRegOut_recalcFC.txt', cell_type_use[cell_num],disease_use[disease_num]))
        if (file.exists(file_name)) {
            print(file_name)
            res <- read.table(file_name, sep = '\t', header = TRUE)
            pval_col <- paste0('pvalue_',disease_use[disease_num])
            if(pval_col %in% colnames(res)){
                logfc_col <- 'L2FC'
                res$rank = (-log10(as.numeric(res[,pval_col])))*res[,logfc_col]
                res <- res %>% 
                          filter_all(all_vars(!is.infinite(.)))
                res = data.frame("SYMBOL" = res$peak,
                             "stat" = res$rank)
                #res$SYMBOL <- paste("chr", res$SYMBOL, sep = "")
    
                res = res[!grepl(pattern = "NA", x = res$SYMBOL),]
                ranks <- deframe(res)
                ranks = sort(ranks, decreasing=TRUE)
                deseq_df <- data.frame(gene = res$SYMBOL, GSEArank = res$stat)
                KEGG_react_fgseaRes <- fgseaMultilevel(pathways=KEGG_react,
                                       stats=ranks,
                                       eps =0.0,
                                       minSize  = 0)#,nPermSimple = 10000)
               message("Number of total enriched terms ", cell_type_use[cell_num], " for ", disease_use[disease_num] ,": ", nrow(KEGG_react_fgseaRes))
               FDR_tresh = 0.10
               KEGG_react_fgseaRes.tresh = KEGG_react_fgseaRes[KEGG_react_fgseaRes$padj < FDR_tresh,]
               message("Number of significant terms: ", cell_type_use[cell_num], ": ",nrow(KEGG_react_fgseaRes.tresh))
               ## Add categories
               res <- KEGG_react_fgseaRes.tresh
               top10_up <- res[which(res$ES > 0),][c(1:10),]
               top10_down <- res[which(res$ES < 0),][c(1:10),]
               top10_up$LOG10P <- -log10(top10_up$pval)
               top10_down$LOG10P <- -log10(top10_down$pval)
               outfile_all <-  paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_fgsea_allResults.tsv')
               KEGG_react_fgseaRes %>% 
                   rowwise() %>% 
                   mutate_if(is.list, ~paste(unlist(.), collapse = ',')) %>% 
                   write.table(outfile_all,sep='\t',quote = FALSE, row.names = FALSE)
               if (nrow(res) > 0){
                   outfile_sig <-  paste0(cell_type_use[cell_num],"_",disease_use[disease_num],'_fgsea_sigResults.tsv')
                   res %>% 
                       rowwise() %>% 
                       mutate_if(is.list, ~paste(unlist(.), collapse = ',')) %>% 
                       write.table(outfile_sig,sep='\t',quote = FALSE, row.names = FALSE)
                    }
            } else next
        
                
    }
         else {
    # File does not exist, skip to the next iteration
        print(paste("File does not exist:", file_name))
        next
          }
    }
    setwd('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/GRN/CleanUpNotebooks/FilterGRNs/fGSEA_TF2peaks/')

}

In [2]:
sessionInfo()

R version 4.1.1 (2021-08-10)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 20.04.2 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.9.0
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.9.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] stats4    grid      stats     graphics  grDevices utils     datasets 
[8] methods   base     

other attached packages:
 [1] tictoc_1.2           org.Hs.eg.db_3.14.0  matrixStats_1.2.0   
 [4] edgeR_3.36.0         limma_3.50.3         GSVA_1.42.0         
 [7] GSEABase_1.56.0      graph_1.72.0         annotate_1.72.0     
[10] XML_3.99-0.14        AnnotationDbi_